# 🔢 InklusiKerja — Step 2: Embedding & FAISS Indexing (v3)

Notebook ini membangun vector index dari data jobs yang sudah diproses.

**Yang dilakukan:**
- Load model sentence-transformer (local fine-tuned atau HuggingFace)
- Encode document_text jobs → embedding vector
- Build FAISS IndexFlatIP untuk pencarian semantik
- Simpan index, metadata, dan config

**Prerequisite:** Jalankan `01_preprocessing.ipynb` terlebih dahulu

**Output:** `data/index/jobs.faiss`, `data/index/jobs_metadata.pkl`, `data/index/job_embeddings.npy`, `data/index/config.json`

## 1. Import & Konfigurasi

In [ ]:
import os
import json
import time
import pickle
import ast
import numpy as np
import pandas as pd
from pathlib import Path

PROCESSED_DIR = "data/processed"
INDEX_DIR     = "data/index"
os.makedirs(INDEX_DIR, exist_ok=True)

# Pilih model: 'multilingual_minilm', 'indobert_semantic', 'multilingual_mpnet',
# atau path ke model local (misal 'models/finetuned')
SELECTED_MODEL = "models/finetuned"  # Ganti jika tidak punya model fine-tuned

MODEL_OPTIONS = {
    "multilingual_minilm": {
        "name": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        "dim" : 384,
        "notes": "Cepat, multilingual, cocok untuk prototyping.",
    },
    "indobert_semantic": {
        "name": "LazarusNLP/indobert-base-p2",
        "dim" : 768,
        "notes": "Fine-tuned untuk semantic similarity Bahasa Indonesia. REKOMENDASI.",
    },
    "multilingual_mpnet": {
        "name": "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
        "dim" : 768,
        "notes": "Lebih akurat dari MiniLM, masih multilingual.",
    },
}

REQUIRED_COLUMNS = [
    "Jenis Disabilitas", "Kebutuhan Aksesibilitas", "Job Title",
    "Level", "Deskripsi Kualifikasi", "job_id", "skill_tags", "level_rank",
]

print("✅ Konfigurasi selesai")
print(f"   Processed Dir : {PROCESSED_DIR}")
print(f"   Index Dir     : {INDEX_DIR}")
print(f"   Selected Model: {SELECTED_MODEL}")

## 2. Load Data Jobs yang Sudah Diproses

In [ ]:
jobs_df = pd.read_csv(f"{PROCESSED_DIR}/jobs_processed.csv")
print(f"📊 {len(jobs_df)} pekerjaan dimuat dari CSV")

# Validasi kolom wajib
missing_cols = [c for c in REQUIRED_COLUMNS if c not in jobs_df.columns]
if missing_cols:
    raise ValueError(f"❌ Kolom tidak ditemukan: {missing_cols}")
print("   ✓ Semua kolom wajib tersedia")

print(f"\n   Distribusi Jenis Disabilitas:")
for dt, count in jobs_df["Jenis Disabilitas"].value_counts().items():
    print(f"      {count:4d}x  {dt}")

jobs_df[["job_id", "Job Title", "Jenis Disabilitas", "Level"]].head(5)

## 3. Inisialisasi Embedding Engine

In [ ]:
from sentence_transformers import SentenceTransformer

class EmbeddingEngine:
    def __init__(self, model_key: str = SELECTED_MODEL):
        if os.path.isdir(model_key):
            self.model_name = model_key
            self.dim = 384
            print(f"🤖 Loading local fine-tuned model: {self.model_name}")
        else:
            model_cfg = MODEL_OPTIONS[model_key]
            self.model_name = model_cfg["name"]
            self.dim = model_cfg["dim"]
            print(f"🤖 Loading model: {self.model_name}")
            print(f"   Catatan: {model_cfg['notes']}")

        print(f"   Dimensi embedding: {self.dim}")
        self.model = SentenceTransformer(self.model_name)
        print("✅ Model berhasil dimuat!")

    def encode(self, texts, batch_size=32, show_progress=True, normalize=True):
        print(f"\n🔄 Encoding {len(texts)} dokumen...")
        start = time.time()
        embeddings = self.model.encode(
            texts, batch_size=batch_size,
            show_progress_bar=show_progress,
            convert_to_numpy=True,
            normalize_embeddings=normalize,
        )
        elapsed = time.time() - start
        print(f"✅ Selesai dalam {elapsed:.1f}s | Shape: {embeddings.shape}")
        return embeddings.astype(np.float32)

engine = EmbeddingEngine(model_key=SELECTED_MODEL)

## 4. Build Metadata & Document Text

In [ ]:
def parse_skill_tags(val) -> list:
    if isinstance(val, list): return val
    if pd.isna(val) or val in ("", "[]"):  return []
    try: return ast.literal_eval(str(val))
    except: return []

def build_document_text(row) -> str:
    """Format natural language v3 — lebih konsisten dengan query."""
    disability    = row.get("Jenis Disabilitas", "")
    job_title     = row.get("Job Title", "")
    level         = row.get("Level", "")
    qualification = row.get("Deskripsi Kualifikasi", "")
    accessibility = row.get("Kebutuhan Aksesibilitas", "")
    skill_tags    = parse_skill_tags(row.get("skill_tags", []))

    text = (
        f"Lowongan pekerjaan {job_title} untuk penyandang {disability}. "
        f"Level: {level}. "
        f"Kualifikasi: {qualification} "
    )
    if skill_tags:
        text += f"Skill yang dibutuhkan: {', '.join(skill_tags)}. "
    if accessibility:
        text += f"Aksesibilitas yang disediakan: {accessibility}."
    return text

# Build metadata dict & kumpulkan texts untuk embedding
metadata = {}
texts_to_embed = []

for idx, row in jobs_df.iterrows():
    skill_tags = parse_skill_tags(row.get("skill_tags", []))
    level_rank = row.get("level_rank", 1)
    level_rank = int(level_rank) if pd.notna(level_rank) else 1

    metadata[idx] = {
        "job_id"         : row["job_id"],
        "job_title"      : row["Job Title"],
        "disability_type": row["Jenis Disabilitas"],
        "level"          : row["Level"],
        "level_rank"     : level_rank,
        "qualification"  : row["Deskripsi Kualifikasi"],
        "accessibility"  : row["Kebutuhan Aksesibilitas"],
        "skill_tags"     : skill_tags,
    }
    texts_to_embed.append(build_document_text(row))

with_tags = sum(1 for v in metadata.values() if v["skill_tags"])
print(f"✅ Metadata untuk {len(metadata)} pekerjaan siap")
print(f"   → {with_tags} jobs memiliki skill_tags ({with_tags/len(metadata)*100:.0f}%)")
print(f"\n   Preview document text (baris pertama):")
print(f"   {texts_to_embed[0][:150]}...")

## 5. Encode Dokumen → Embedding

In [ ]:
embeddings = engine.encode(texts_to_embed, batch_size=64)
print(f"\nEmbedding shape: {embeddings.shape}")
print(f"Dtype: {embeddings.dtype}")
print(f"Sample norm (baris 0): {np.linalg.norm(embeddings[0]):.4f}  (harus ≈ 1.0 jika normalized)")

## 6. Build FAISS Index

In [ ]:
import faiss

# IndexFlatIP — Inner Product (setara cosine jika vektor ternormalisasi)
dim   = engine.dim
index = faiss.IndexFlatIP(dim)
index.add(embeddings)
print(f"✅ IndexFlatIP dibangun | {index.ntotal} vektor | dim={dim}")

# Test quick search
test_vec = embeddings[0:1]
scores, indices = index.search(test_vec, 3)
print(f"\n🔍 Test search (query = dokumen ke-0):")
for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), 1):
    print(f"   #{rank} → idx={idx} | score={score:.4f} | {metadata[idx]['job_title']}")

## 7. Simpan Semua Artifacts

In [ ]:
# Simpan FAISS index
faiss_path = f"{INDEX_DIR}/jobs.faiss"
faiss.write_index(index, faiss_path)
print(f"💾 FAISS index → {faiss_path}")

# Simpan raw embeddings
emb_path = f"{INDEX_DIR}/job_embeddings.npy"
np.save(emb_path, embeddings)
print(f"💾 Raw embeddings → {emb_path}")

# Simpan metadata
meta_path = f"{INDEX_DIR}/jobs_metadata.pkl"
with open(meta_path, "wb") as f:
    pickle.dump(metadata, f)
print(f"💾 Metadata → {meta_path}")

# Simpan config
config = {
    "model_name"     : engine.model_name,
    "embedding_dim"  : engine.dim,
    "total_jobs"     : len(jobs_df),
    "index_type"     : "IndexFlatIP",
    "normalized"     : True,
    "has_skill_tags" : True,
    "document_format": "natural_language_v3",
}
config_path = f"{INDEX_DIR}/config.json"
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)
print(f"💾 Config → {config_path}")

print("\n" + "=" * 50)
print("✅ Build selesai! Semua file tersimpan di data/index/")
print("   → jobs.faiss")
print("   → jobs_metadata.pkl")
print("   → job_embeddings.npy")
print("   → config.json")
print("\n➡️  Lanjut ke 03_recommendation_engine.ipynb")